# scrape judul dan url


In [12]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd

async def scrape():

    data = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=False)

        page = await browser.new_page()

        await page.goto(
            "https://malangtimes.com/kanal/pendidikan",
            wait_until="networkidle"
        )

        await page.locator("#load-more").click()
        await page.wait_for_timeout(2000)

        await page.locator("#load-more").click()
        await page.wait_for_timeout(2000)

        soup = BeautifulSoup(
            await page.content(),
            "html.parser"
        )

        for a in soup.select("a[href*='/baca/']"):

            url = a.get("href")

            title_tag = a.select_one("h3")

            if not url or not title_tag:
                continue

            data.append({
                "title": title_tag.get_text(strip=True),
                "url": url
            })

        await browser.close()

    df = pd.DataFrame(data).drop_duplicates()

    print(df.head())

    df.to_csv(
        "malangtimes_pendidikan.csv",
        index=False
    )

    print(f"saved {len(df)} articles")

    return df

df = await scrape()

                                               title  \
0  SPMB Jatim Tahap II 2026 Dibuka: Cek Jadwal, S...   
1  Kabar Baik! Pendaftaran Offline SPMB Situbondo...   
2  Waspada Helikopter Parenting, Saat Kasih Sayan...   
3  Magang Nasional Kemnaker 2026 Angkatan 2 Seger...   
4  Perkuat Jejaring Eropa, Unisma Bidik Kolaboras...   

                                                 url  
0  https://malangtimes.com/baca/3331345536/202606...  
1  https://malangtimes.com/baca/3331345532/202606...  
2  https://malangtimes.com/baca/3331345525/202606...  
3  https://malangtimes.com/baca/3331345505/202606...  
4  https://malangtimes.com/baca/3331345504/202606...  
saved 28 articles


# scrape isi url 

In [2]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd

async def scrape_articles():

    urls_df = pd.read_csv("csv/malangtimes_pendidikan.csv")

    articles = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=False)

        page = await browser.new_page()

        for idx, row in urls_df.iterrows():

            url = row["url"]

            try:

                await page.goto(
                    url,
                    wait_until="networkidle"
                )

                soup = BeautifulSoup(
                    await page.content(),
                    "html.parser"
                )

                # Judul
                title = ""
                title_tag = soup.select_one("h1")

                if title_tag:
                    title = title_tag.get_text(strip=True)
                
                # Published date
                published_date = None

                date_tag = soup.select_one("p.float-right.text-muted")

                if date_tag:
                    published_date = date_tag.get_text(strip=True)

                # Isi berita
                content = ""
                content_tag = soup.select_one(
                    "#appCapsule > div:nth-child(6) > div.col-12.col-lg-8 > div.blog-post.mt-1 > div.post-body"
                )

                if content_tag:
                    content = content_tag.get_text(
                        separator="\n",
                        strip=True
                    )

                # Penulis & Editor
                author = None
                editor = None

                text = soup.get_text(" ", strip=True)

                import re

                author_match = re.search(
                    r"Penulis\s*:\s*(.*?)\s*-\s*Editor",
                    text
                )

                editor_match = re.search(
                    r"Editor\s*:\s*(.*?)(?:Baca Juga|$)",
                    text
                )

                if author_match:
                    author = author_match.group(1).strip()

                if editor_match:
                    editor = editor_match.group(1).strip()

                articles.append({
                    "title": title,
                    "published_date": published_date,
                    "content": content,
                    "author": author,
                    "editor": editor,
                    "source": "malangtimes",
                    "url": url
                })

                print(
                    f"[{idx+1}/{len(urls_df)}] success"
                )

            except Exception as e:

                print(
                    f"[{idx+1}/{len(urls_df)}] failed: {e}"
                )

        await browser.close()

    df = pd.DataFrame(articles)

    df.to_csv(
        "malangtimes_articles.csv",
        index=False
    )

    print(
        f"saved {len(df)} articles"
    )

    return df


df_articles = await scrape_articles()

[1/28] success
[2/28] success
[3/28] success
[4/28] success
[5/28] success
[6/28] success
[7/28] success
[8/28] success
[9/28] success
[10/28] success
[11/28] failed: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://malangtimes.com/baca/3331345440/20260615/035900/unisma-perluas-kolaborasi-global-jajaki-riset-bersama-dan-mobilitas-akademik-dengan-kings-college-london", waiting until "networkidle"

[12/28] success
[13/28] success
[14/28] success
[15/28] success
[16/28] success
[17/28] success
[18/28] success
[19/28] success
[20/28] success
[21/28] success
[22/28] success
[23/28] success
[24/28] success
[25/28] success
[26/28] success
[27/28] success
[28/28] success
saved 27 articles
